In [3]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Download NLTK resources
nltk.download('stopwords')
nltk.download('punkt')

# ==========================================
# TASK 1: DATASET UNDERSTANDING
# ==========================================
print("--- Task 1: Dataset Understanding ---")
df = pd.read_csv(r'C:\Users\swaro\Downloads\ai_project_synthetic_datasets-20260517T141911Z-3-001\ai_project_synthetic_datasets\part_3_nlp_sequence_modeling\customer_support_text_classification.csv')

num_records = len(df)
target_classes = df['sentiment_label'].unique()
avg_word_count = df['word_count'].mean()
class_distribution = df['sentiment_label'].value_counts(normalize=True) * 100

print(f"Number of records: {num_records}")
print(f"Target classes: {target_classes}")
print(f"Average text length (words): {avg_word_count:.2f}\n")
print("Class Distribution (%):")
print(class_distribution.to_string())
print("\nSample Text Records:")
print(df[['customer_message', 'sentiment_label']].head(3))

# ==========================================
# TASK 2: TEXT PREPROCESSING
# ==========================================
print("\n--- Task 2: Text Preprocessing ---")
stop_words = set(stopwords.words('english'))

def clean_text(text):
    if not isinstance(text, str):
        return ""
    # Lowercasing
    text = text.lower()
    # Remove ticket numbers specifically (e.g., "ticket number is 12345")
    text = re.sub(r'ticket number is \d+', '', text)
    # Remove numbers and special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Tokenization and Stopword Removal
    tokens = text.split()
    cleaned_tokens = [word for word in tokens if word not in stop_words]
    return " ".join(cleaned_tokens)

df['cleaned_message'] = df['customer_message'].apply(clean_text)

# Encode Target Labels
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['sentiment_label']) # negative=0, neutral=1, positive=2

# Train-Test Split
X_train, X_val, y_train, y_val = train_test_split(
    df['cleaned_message'], df['label'], test_size=0.2, random_state=42, stratify=df['label']
)
print("Preprocessing complete. Sample cleaned message:")
print(f"Original: {df['customer_message'].iloc[0]}")
print(f"Cleaned : {df['cleaned_message'].iloc[0]}")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\swaro\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


--- Task 1: Dataset Understanding ---
Number of records: 1500
Target classes: <StringArray>
['neutral', 'positive', 'negative']
Length: 3, dtype: str
Average text length (words): 12.72

Class Distribution (%):
sentiment_label
neutral     34.933333
negative    33.133333
positive    31.933333

Sample Text Records:
                                    customer_message sentiment_label
0  I need information about the payment process. ...         neutral
1      I need information about the payment process.         neutral
2  The refund process was fast and convenient. I ...        positive

--- Task 2: Text Preprocessing ---
Preprocessing complete. Sample cleaned message:
Original: I need information about the payment process. My ticket number is 78732. Please respond as soon as possible.
Cleaned : need information payment process please respond soon possible


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\swaro\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [2]:
pip install nltk

  Obtaining dependency information for nltk from https://files.pythonhosted.org/packages/9d/91/04e965f8e717ba0ab4bdca5c112deeab11c9e750d94c4d4602f050295d39/nltk-3.9.4-py3-none-any.whl.metadata
  Obtaining dependency information for click from https://files.pythonhosted.org/packages/ee/ae/8e92f8058baf87f6c7d86ee7e457668690195cc77efedb8d3797a06e3940/click-8.4.0-py3-none-any.whl.metadata
  Obtaining dependency information for regex>=2021.8.3 from https://files.pythonhosted.org/packages/78/87/240d36864f9e48ace85f72e79ced97ceb7f27ce87739a947dcb834b4e6bc/regex-2026.5.9-cp312-cp312-win_amd64.whl.metadata
     ---------------------------------------- 0.0/41.5 kB ? eta -:--:--
     ---------------------------------------- 0.0/41.5 kB ? eta -:--:--
     ---------------------------------------- 0.0/41.5 kB ? eta -:--:--
     ---------------------------------------- 41.5/41.5 kB 1.0 MB/s eta 0:00:00
  Obtaining dependency information for tqdm from https://files.pythonhosted.org/packages/16/e1/3079

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# ==========================================
# TASK 3: TEXT VECTORIZATION (TF-IDF)
# ==========================================
tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)

# ==========================================
# TASK 4: BASELINE MODEL
# ==========================================
print("\n--- Task 4: Baseline Model (Logistic Regression) ---")
baseline_model = LogisticRegression(max_iter=1000, random_state=42)
baseline_model.fit(X_train_tfidf, y_train)

# Evaluation
baseline_preds = baseline_model.predict(X_val_tfidf)
baseline_acc = accuracy_score(y_val, baseline_preds)

print(f"Baseline Validation Accuracy: {baseline_acc:.4f}\n")
print("Classification Report:")
print(classification_report(y_val, baseline_preds, target_names=label_encoder.classes_))

# Save results for the repository requirements
import os
os.makedirs('results', exist_ok=True)
report_df = pd.DataFrame(classification_report(y_val, baseline_preds, target_names=label_encoder.classes_, output_dict=True)).transpose()
report_df.to_csv('results/model_evaluation.csv', index=True)


--- Task 4: Baseline Model (Logistic Regression) ---
Baseline Validation Accuracy: 1.0000

Classification Report:
              precision    recall  f1-score   support

    negative       1.00      1.00      1.00        99
     neutral       1.00      1.00      1.00       105
    positive       1.00      1.00      1.00        96

    accuracy                           1.00       300
   macro avg       1.00      1.00      1.00       300
weighted avg       1.00      1.00      1.00       300

